This notebook contains the code to prepare SPRNSC RT data aligned with LM surprisals for statistical analysis.

SPRNSC shares the same text as MazeNSC. Before running this notebook, make sure you have already:
1. Generated surprisals from MazeNSC (see the folder "MazeNSC").
1. Downloaded the SPRNSC RT file from [this google drive folder](https://drive.google.com/drive/folders/1hjIIgxO5LfufE07D9bzazIOwqyfRRy6d?usp=sharing). Note that the RT file from this folder is different from the one newly released [here](https://github.com/languageMIT/naturalstories/tree/master/naturalstories_RTS). The original RT file we used contains an alignment error, which we have fixed in our own R analysis code.

In [1]:
import pandas as pd
import wordfreq
import string

In [2]:
NSC_RT_FILE = "corpus_data/SPR_processed_RTs.tsv"
NSC_LM_FILE = "corpus_data/NSC_lm_features.csv"
df_RT = pd.read_csv(NSC_RT_FILE, sep='\t')
df_NSC_lm = pd.read_csv(NSC_LM_FILE)

In [3]:
df_NSC_lm.head(5)

,global_word_id,story_id,sentence_id,word_id,word,sentence,hftoken,token_id,attn_idx,entropy_base_gpt2,...,entropy_lambda0p0,surp_lambda0p0,entropy_lambda0p001,surp_lambda0p001,entropy_lambda0p01,surp_lambda0p01,entropy_lambda0p1,surp_lambda0p1,entropy_lambda1p0,surp_lambda1p0
0,0,1,1,0,If,If you were to journey to the North of England...,ĠIf,1002,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,1,1,you,If you were to journey to the North of England...,Ġyou,345,1,5.166911,...,4.752929,2.208604,4.560232,3.636842,5.553502,6.756579,5.255672,4.951382,4.830709,3.712671
2,2,1,1,2,were,If you were to journey to the North of England...,Ġwere,547,2,4.898644,...,5.177368,7.297988,4.729102,6.372193,4.761328,5.598334,4.756964,5.780215,5.394075,5.777753
3,3,1,1,3,to,If you were to journey to the North of England...,Ġto,284,3,6.098342,...,5.843816,3.290146,6.322002,4.708209,6.233095,6.080613,6.055978,5.225512,6.248493,4.751654
4,4,1,1,4,journey,If you were to journey to the North of England...,Ġjourney,7002,4,5.879211,...,5.766353,16.027717,5.763282,15.917110,5.475807,14.754842,5.654287,16.823480,5.916594,14.541185


In [4]:
model_names = ['base_gpt2', 'lambda0p0', 'lambda0p001', 'lambda0p01', 'lambda0p1', 'lambda1p0']

def align_features(df, model_names):
    '''
    Align gpt2 tokenization with the tokenization of behavioral data.
    '''
    word_features_dict = {'global_word_id': df.global_word_id.iloc[0], 
                          'story_id': df.story_id.iloc[0],
                          'sentence_id': df.sentence_id.iloc[0],
                          'word_id': df.word_id.iloc[0], 
                          'word': df.word.iloc[0], 
                          'sentence': df.sentence.iloc[0]}
    for model_name in model_names:
        word_features_dict[f'word_surp_{model_name}'] = df[f'surp_{model_name}'].sum()
    result = pd.DataFrame([word_features_dict])

    return result

# Align gpt2 tokenization with words
df_NSC_lm = df_NSC_lm.groupby("global_word_id", group_keys=False).apply(
        lambda group: align_features(group, model_names)
        ).reset_index(drop=True)

df_NSC_lm.head(15)

/var/folders/4k/_kj6xmhj7tbbbj61g0kc59280000gn/T/ipykernel_7778/375583256.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_NSC_lm = df_NSC_lm.groupby("global_word_id", group_keys=False).apply(


,global_word_id,story_id,sentence_id,word_id,word,sentence,word_surp_base_gpt2,word_surp_lambda0p0,word_surp_lambda0p001,word_surp_lambda0p01,word_surp_lambda0p1,word_surp_lambda1p0
0,0,1,1,0,If,If you were to journey to the North of England...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1,1,1,1,you,If you were to journey to the North of England...,3.940546,2.208604,3.636842,6.756579,4.951382,3.712671
2,2,1,1,2,were,If you were to journey to the North of England...,6.965252,7.297988,6.372193,5.598334,5.780215,5.777753
3,3,1,1,3,to,If you were to journey to the North of England...,5.014768,3.290146,4.708209,6.080613,5.225512,4.751654
4,4,1,1,4,journey,If you were to journey to the North of England...,16.781034,16.027717,15.917110,14.754842,16.823480,14.541185
5,5,1,1,5,to,If you were to journey to the North of England...,2.785678,2.722056,1.713383,2.705173,2.307931,2.703549
6,6,1,1,6,the,If you were to journey to the North of England...,2.139630,2.218693,1.741782,1.776879,1.821066,1.826238
7,7,1,1,7,North,If you were to journey to the North of England...,7.976809,6.667486,7.905368,7.620416,7.366017,6.480740
8,8,1,1,8,of,If you were to journey to the North of England...,5.259198,9.060716,6.954817,4.616590,5.944073,4.826832
9,9,1,1,9,"England,",If you were to journey to the North of England...,20.422023,26.089550,24.067880,26.170974,24.628530,22.410448


In [5]:
df_NSC_lm["zone"] = df_NSC_lm.groupby("story_id").cumcount() + 1

In [6]:
df_NSC_lm.tail(5)

,global_word_id,story_id,sentence_id,word_id,word,sentence,word_surp_base_gpt2,word_surp_lambda0p0,word_surp_lambda0p001,word_surp_lambda0p01,word_surp_lambda0p1,word_surp_lambda1p0,zone
10251,10251,10,43,11,and,The hope is that this research leads to better...,3.081454,2.334810,2.408892,3.076727,3.287968,3.573369,935
10252,10252,10,43,12,better,The hope is that this research leads to better...,3.747456,6.685374,5.968991,8.063255,6.189188,7.577277,936
10253,10253,10,43,13,treatments,The hope is that this research leads to better...,5.583406,10.038446,9.444727,10.816152,8.341884,9.813012,937
10254,10254,10,43,14,for,The hope is that this research leads to better...,1.611072,2.043128,3.126701,1.978742,2.011098,3.187478,938
10255,10255,10,43,15,Tourette's.,The hope is that this research leads to better...,16.954564,39.587301,45.944649,42.070122,41.081674,42.486754,939


In [7]:
df_RT.head(5)

,WorkerId,WorkTimeInSeconds,correct,item,zone,RT,word,nItem,meanItemRT,sdItemRT,gmeanItemRT,gsdItemRT
0,A3QJPB0NZU5PY1,3960,6,1,1,924,If,84,369.011905,160.579935,340.566023,1.490513
1,A2RPQGUWVZPX7U,2431,5,1,1,474,If,84,369.011905,160.579935,340.566023,1.490513
2,A11KMPAZSE5Q0Q,1287,5,1,1,272,If,84,369.011905,160.579935,340.566023,1.490513
3,A1U1QL617G5DU3,2074,6,1,1,354,If,84,369.011905,160.579935,340.566023,1.490513
4,ACTW5YEWV9OR0,2213,6,1,1,577,If,84,369.011905,160.579935,340.566023,1.490513


In [8]:
df_RT = df_RT.rename(columns={
    "WorkerId": "subject",
    "item": "story_id",
})

In [9]:
df_RT.head(5)

,subject,WorkTimeInSeconds,correct,story_id,zone,RT,word,nItem,meanItemRT,sdItemRT,gmeanItemRT,gsdItemRT
0,A3QJPB0NZU5PY1,3960,6,1,1,924,If,84,369.011905,160.579935,340.566023,1.490513
1,A2RPQGUWVZPX7U,2431,5,1,1,474,If,84,369.011905,160.579935,340.566023,1.490513
2,A11KMPAZSE5Q0Q,1287,5,1,1,272,If,84,369.011905,160.579935,340.566023,1.490513
3,A1U1QL617G5DU3,2074,6,1,1,354,If,84,369.011905,160.579935,340.566023,1.490513
4,ACTW5YEWV9OR0,2213,6,1,1,577,If,84,369.011905,160.579935,340.566023,1.490513


In [10]:
result_df = pd.merge(df_RT, df_NSC_lm, how='inner', on=['story_id', 'zone'])
print(result_df.shape)
result_df = result_df.rename(columns={"word_x": "word", 
                                      "word_y": "word_lm"})
print(result_df.shape)
result_df.head()

(848767, 23)
(848767, 23)


,subject,WorkTimeInSeconds,correct,story_id,zone,RT,word,nItem,meanItemRT,sdItemRT,...,sentence_id,word_id,word_lm,sentence,word_surp_base_gpt2,word_surp_lambda0p0,word_surp_lambda0p001,word_surp_lambda0p01,word_surp_lambda0p1,word_surp_lambda1p0
0,A3QJPB0NZU5PY1,3960,6,1,1,924,If,84,369.011905,160.579935,...,1,0,If,If you were to journey to the North of England...,0.0,0.0,0.0,0.0,0.0,0.0
1,A2RPQGUWVZPX7U,2431,5,1,1,474,If,84,369.011905,160.579935,...,1,0,If,If you were to journey to the North of England...,0.0,0.0,0.0,0.0,0.0,0.0
2,A11KMPAZSE5Q0Q,1287,5,1,1,272,If,84,369.011905,160.579935,...,1,0,If,If you were to journey to the North of England...,0.0,0.0,0.0,0.0,0.0,0.0
3,A1U1QL617G5DU3,2074,6,1,1,354,If,84,369.011905,160.579935,...,1,0,If,If you were to journey to the North of England...,0.0,0.0,0.0,0.0,0.0,0.0
4,ACTW5YEWV9OR0,2213,6,1,1,577,If,84,369.011905,160.579935,...,1,0,If,If you were to journey to the North of England...,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
# Add word frequency
def get_frequency(word, language='en'):
	return wordfreq.zipf_frequency(word, language)

result_df['logfreq'] = result_df.apply(
	lambda df: get_frequency(df['word'], 'en'), axis=1
	)

In [12]:
# Mark punctuation
result_df['has_punct'] = result_df['word'].str.contains(rf"[{string.punctuation}]", regex=True)

In [13]:
result_df.to_csv('corpus_data/SPRNSC_RT_lm_features.csv', index=False)